In [ ]:
"""
Add description of project here
"""


### Installing needed packages

In [ ]:
#pip install langchain-cohere

In [ ]:
# https://github.com/googlecolab/colabtools/issues/5455
# For langchain-cohere==0.4.4 downgrade cohere to 5.15.0, it will solve the problem.

#%pip uninstall cohere
#%pip install cohere==5.15.0

### Import packages and env vars

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  # Loads from .env
cohere_api_key=os.getenv("COHERE_TOKEN")

In [ ]:
from langchain_cohere import ChatCohere
from langchain_core.messages import AIMessage, HumanMessage

In [ ]:
# Memory start
from langchain.memory import ConversationSummaryMemory
from langchain import PromptTemplate
from langchain import LLMChain


### Define arguments for chatbot

In [4]:
# Define the Cohere LLM
llm = ChatCohere(
    cohere_api_key=cohere_api_key, model="command-a-03-2025"
)

In [ ]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)


In [ ]:

# Create a summary prompt template
# I dont know if the tokens <s><|user|> are useful for this model
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""

summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)



C:\Users\Carlos Ivan\AppData\Local\Temp\ipykernel_13752\2483041381.py:39: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  llm_chain = LLMChain(


In [ ]:
# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)



### Define chain

In [ ]:
# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

### Test it

In [15]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Carlos. What is 1 + 4?"})


{'input_prompt': 'Hi! My name is Carlos. What is 1 + 4?',
 'chat_history': '',
 'text': 'Hi Carlos! Nice to meet you. The answer to 1 + 4 is **5**. How can I assist you further?'}

In [16]:
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': '**New summary:**\n\nThe conversation begins with Carlos introducing himself and asking a simple math question: "What is 1 + 4?" The AI responds by greeting Carlos, providing the correct answer (**5**), and offering further assistance.  \n\n**Updated summary:**\n\nThe conversation starts with Carlos introducing himself and asking a basic math question, "What is 1 + 4?" The AI greets Carlos, correctly answers **5**, and inquires how it can assist him further.',
 'text': 'Your name is **Carlos**.'}

In [17]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': '**New summary:**\n\nThe conversation begins with Carlos introducing himself and asking a simple math question: "What is 1 + 4?" The AI responds by greeting Carlos, providing the correct answer (**5**), and offering further assistance. Carlos then asks, "What is my name?" The AI correctly identifies his name as **Carlos**.  \n\n**Updated summary:**\n\nThe conversation starts with Carlos introducing himself and asking a basic math question, "What is 1 + 4?" The AI greets Carlos, correctly answers **5**, and inquires how it can assist him further. Carlos follows up by asking, "What is my name?" The AI accurately responds with his name, **Carlos**.',
 'text': 'The first question you asked was, "What is 1 + 4?"'}

### Access memory

In [18]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': '**New summary:**\n\nThe conversation begins with Carlos introducing himself and asking a basic math question, "What is 1 + 4?" The AI greets Carlos, correctly answers **5**, and inquires how it can assist him further. Carlos follows up by asking, "What is my name?" The AI accurately responds with his name, **Carlos**. Later, Carlos asks, "What was the first question I asked?" The AI correctly recalls and answers, "The first question you asked was, \'What is 1 + 4?\'"'}